# Forge - Notebook 03

# Document Chunking

This notebook splits processed documents into semantically meaningful chunks that are optimized for retrieval.

Output:
- Chunked JSON files
- Chunking statistics
- Chunking report

In [2]:
!pip install -q langchain-text-splitters

In [3]:
from pathlib import Path
import json
from uuid import uuid4

import pandas as pd

from google.colab import drive

from langchain_text_splitters import RecursiveCharacterTextSplitter

In [4]:
drive.mount("/content/drive")

Mounted at /content/drive


In [8]:
CONFIG["chunks"].mkdir(parents=True, exist_ok=True)

In [9]:
PROJECT_ROOT = Path("/content/drive/MyDrive/forge")

CONFIG = {
    "project_root": PROJECT_ROOT,
    "processed": PROJECT_ROOT / "knowledge_base" / "processed",
    "chunks": PROJECT_ROOT / "knowledge_base" / "chunks",
}

In [10]:
for name, path in CONFIG.items():
    print(f"{name:15} : {'✓' if path.exists() else '✗'}")

project_root    : ✓
processed       : ✓
chunks          : ✓


In [11]:
processed_files = sorted(CONFIG["processed"].rglob("*.json"))

print(f"Processed Documents: {len(processed_files)}")

Processed Documents: 283


In [12]:
documents = []

for file_path in processed_files:
    with open(file_path, "r", encoding="utf-8") as file:
        documents.append(json.load(file))

print(f"Loaded {len(documents)} documents.")

Loaded 283 documents.


In [13]:
sample = documents[0]
print(json.dumps(sample, indent=2)[:2500])

{
  "technology": "AWS SageMaker",
  "technology_id": "source-aws-sagemaker",
  "category": "deployment",
  "organization": "Amazon Web Services",
  "license": "",
  "priority": "high",
  "update_frequency": "weekly",
  "source": "official_documentation",
  "url": "https://docs.aws.amazon.com/sagemaker/",
  "content": "Use Machine Learning Environments Learn about machine learning environments that Amazon SageMaker AI offers.\nLabel Data with a Human-in-the-loop Learn how to use a human-in-the-loop to help label data more accurately.\nCreate, Store, and Share Features Learn how to create, store, and share extracted data signals (features) for machine learning.\nUse Docker Containers to Build Models Learn how to use Docker containers to build your machine learning models.\nDetect Bias and Understand Explanations Learn how to detect bias and understand explanations in machine learning models.\nAmazon SageMaker AI Python SDK Use the Amazon SageMaker AI Python SDK library to train and depl

In [14]:
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

print(f"Chunk Size    : {CHUNK_SIZE}")
print(f"Chunk Overlap : {CHUNK_OVERLAP}")

Chunk Size    : 1000
Chunk Overlap : 200


In [15]:
def chunk_document(document):
    chunks = text_splitter.split_text(document["content"])

    chunked_documents = []

    for index, chunk in enumerate(chunks):
        if len(chunk.strip()) < 100:
            continue

        chunked_documents.append({
            "chunk_id": str(uuid4()),
            "technology": document["technology"],
            "technology_id": document["technology_id"],
            "category": document["category"],
            "organization": document["organization"],
            "license": document["license"],
            "priority": document["priority"],
            "update_frequency": document["update_frequency"],
            "source": document["source"],
            "url": document["url"],
            "path": document["path"],
            "chunk_index": index,
            "text": chunk,
            "character_count": len(chunk)
        })

    return chunked_documents

In [16]:
sample_chunks = chunk_document(documents[0])

print(f"Number of Chunks: {len(sample_chunks)}")
print()

print(json.dumps(sample_chunks[0], indent=2)[:2000])

Number of Chunks: 1

{
  "chunk_id": "9bb317d5-269a-42d4-b3f9-53f3f85c18c1",
  "technology": "AWS SageMaker",
  "technology_id": "source-aws-sagemaker",
  "category": "deployment",
  "organization": "Amazon Web Services",
  "license": "",
  "priority": "high",
  "update_frequency": "weekly",
  "source": "official_documentation",
  "url": "https://docs.aws.amazon.com/sagemaker/",
  "path": "aws_sagemaker/official_documentation.json",
  "chunk_index": 0,
  "text": "Use Machine Learning Environments Learn about machine learning environments that Amazon SageMaker AI offers.\nLabel Data with a Human-in-the-loop Learn how to use a human-in-the-loop to help label data more accurately.\nCreate, Store, and Share Features Learn how to create, store, and share extracted data signals (features) for machine learning.\nUse Docker Containers to Build Models Learn how to use Docker containers to build your machine learning models.\nDetect Bias and Understand Explanations Learn how to detect bias and u

In [17]:
all_chunks = []

for document in documents:
    chunks = chunk_document(document)
    all_chunks.extend(chunks)

print(f"Documents : {len(documents)}")
print(f"Chunks    : {len(all_chunks)}")

Documents : 283
Chunks    : 4778


In [18]:
CONFIG["chunks"].mkdir(parents=True, exist_ok=True)

for chunk in all_chunks:
    technology_folder = CONFIG["chunks"] / chunk["technology"]
    technology_folder.mkdir(parents=True, exist_ok=True)

    output_file = technology_folder / f"{chunk['chunk_id']}.json"

    with open(output_file, "w", encoding="utf-8") as file:
        json.dump(chunk, file, indent=2, ensure_ascii=False)

print(f"Saved {len(all_chunks)} chunks.")

Saved 4778 chunks.


In [19]:
combined_file = CONFIG["project_root"] / "knowledge_base" / "chunks.json"

with open(combined_file, "w", encoding="utf-8") as file:
    json.dump(all_chunks, file, indent=2, ensure_ascii=False)

print(f"Saved combined file: {combined_file}")

Saved combined file: /content/drive/MyDrive/forge/knowledge_base/chunks.json


In [20]:
report = pd.DataFrame([
    {
        "chunk_id": chunk["chunk_id"],
        "technology": chunk["technology"],
        "source": chunk["source"],
        "chunk_index": chunk["chunk_index"],
        "characters": chunk["character_count"],
    }
    for chunk in all_chunks
])

report_path = CONFIG["project_root"] / "knowledge_base" / "chunking_report.csv"

report.to_csv(report_path, index=False)

report.head()

,chunk_id,technology,source,chunk_index,characters
0,9660c5d5-5b17-49be-a42f-1d01e3bee92e,AWS SageMaker,official_documentation,0,897
1,a44ad566-bda0-4c6d-b908-a0192b43453d,Anthropic Claude,api_reference,0,930
2,f7b99182-92e4-4455-839a-b9af9af23f59,Anthropic Claude,api_reference,1,874
3,1acf9586-b2ae-443d-b2c3-18b2fb83c4db,Anthropic Claude,api_reference,2,875
4,2237b445-b917-40c2-83ba-09f1c3b44d98,Anthropic Claude,api_reference,3,983


In [21]:
print("=" * 50)
print("DOCUMENT CHUNKING SUMMARY")
print("=" * 50)

print(f"Processed Documents : {len(documents)}")
print(f"Total Chunks        : {len(all_chunks)}")
print(f"Chunk Size          : {CHUNK_SIZE}")
print(f"Chunk Overlap       : {CHUNK_OVERLAP}")

average_size = report["characters"].mean()
min_size = report["characters"].min()
max_size = report["characters"].max()

print(f"Average Chunk Size  : {average_size:.0f} characters")
print(f"Smallest Chunk      : {min_size} characters")
print(f"Largest Chunk       : {max_size} characters")

print(f"\nChunk Directory     : {CONFIG['chunks']}")
print(f"Chunk Report        : {report_path}")

DOCUMENT CHUNKING SUMMARY
Processed Documents : 283
Total Chunks        : 4778
Chunk Size          : 1000
Chunk Overlap       : 200
Average Chunk Size  : 837 characters
Smallest Chunk      : 100 characters
Largest Chunk       : 1000 characters

Chunk Directory     : /content/drive/MyDrive/forge/knowledge_base/chunks
Chunk Report        : /content/drive/MyDrive/forge/knowledge_base/chunking_report.csv


# Conclusion

This notebook successfully transformed the processed documentation corpus into retrieval-ready chunks.

Outputs:
- Chunked JSON files
- Chunking report
- Chunk metadata for vector indexing

These chunks will be embedded and indexed in the next notebook.